# Comparação de modelos candidatos

Notebook integrado da comparação da Seção 4.4 (ART.7). Reúne, num só lugar, todos os modelos
candidatos à predição de Detrator, treinados sobre **a mesma matriz** e avaliados com **as mesmas
métricas e a mesma partição**, para que a equipe compare resultados de forma organizada.

A Seção 5 já instancia Extra Trees como primeiro candidato, mas ainda não o treina nem o registra
para avaliação. As partes comuns (carga, pré-processamento, avaliação e visualização) rodam
mesmo enquanto a lista de modelos treinados está vazia; os demais candidatos serão adicionados
em seus cards.

## Como executar

```bash
pip install -r requirements.txt
jupyter lab notebooks/comparacao_modelos.ipynb
```

Para a execução verificável de ponta a ponta, a partir da raiz do repositório:

```bash
jupyter nbconvert --execute --to notebook --output-dir=.execucao notebooks/comparacao_modelos.ipynb
```

**Pré-requisito**: `data/processed/base_analitica.parquet`, gerada por
`notebooks/pre-processamento.ipynb`. A base não é versionada (o Termo de Abertura veda publicar
dados do parceiro).

**Sem acesso à base real**: gere a base sintética com `python scripts/gerar_dummy.py` e execute
`USAR_BASE_DUMMY=1 jupyter nbconvert --execute --to notebook --output-dir=.execucao notebooks/comparacao_modelos.ipynb`.
Ela serve só para verificar a execução, não para comparar modelos. No Jupyter, inicie o servidor
com `USAR_BASE_DUMMY=1 jupyter lab notebooks/comparacao_modelos.ipynb`.

**Como adicionar um candidato**: crie uma subseção na Seção 5 com justificativa e parâmetros,
instancie o estimador, ajuste-o somente com `X_treino` e `y_treino` e então registre o modelo
treinado em `modelos` com um nome único. As seções 6 e 7 o incluem automaticamente. A seção 5.1
mostra apenas a instanciação: ainda não é um molde completo de treinamento.

Rode as células **em ordem**, de cima para baixo.

## 1. Importações

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
try:
    import sklearn
    from sklearn.ensemble import ExtraTreesClassifier
except ImportError as erro:
    raise ImportError(
        "Instale as dependências com pip install -r requirements.txt para usar Extra Trees."
    ) from erro

# Inclui src/ no caminho de importação tanto no projeto local quanto no Colab.
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

from matriz import preparar_matriz
from modelo import metricas_no_teste

print(f"scikit-learn {sklearn.__version__}")

## 2. Carregamento dos dados

`USAR_BASE_DUMMY = False` carrega a base analítica real, a única que mede o desempenho dos
candidatos. Com `True`, carrega a base sintética de `scripts/gerar_dummy.py`, que serve só para
verificar que o notebook roda sem o dado do parceiro: as métricas dela não valem para a comparação.

In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

USAR_BASE_DUMMY = os.environ.get("USAR_BASE_DUMMY", "0") == "1"

if USAR_BASE_DUMMY:
    CAMINHOS_POSSIVEIS = [
        Path("data/dummy/base_analitica_dummy.parquet"),
        Path("../data/dummy/base_analitica_dummy.parquet"),
    ]
    instrucao = "Gere a base com: python scripts/gerar_dummy.py"
else:
    CAMINHOS_POSSIVEIS = [
        Path("data/processed/base_analitica.parquet"),
        Path("../data/processed/base_analitica.parquet"),
        Path("/content/data/processed/base_analitica.parquet"),
        Path("/content/drive/MyDrive/projeto-azul/data/processed/base_analitica.parquet"),
    ]
    instrucao = (
        "Rode notebooks/pre-processamento.ipynb antes. Para só verificar que o notebook "
        "roda, sem o dado do parceiro, defina USAR_BASE_DUMMY=1 no ambiente."
    )

caminho_base = next((c for c in CAMINHOS_POSSIVEIS if c.exists()), None)
if caminho_base is None:
    raise FileNotFoundError(f"{CAMINHOS_POSSIVEIS[0].name} não encontrada. {instrucao}")

df = pd.read_parquet(caminho_base)
print(f"{caminho_base}: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
if USAR_BASE_DUMMY:
    print("ATENÇÃO: base sintética. As métricas deste notebook não valem para a comparação.")

## 3. Pré-processamento

`preparar_matriz`, de `src/matriz.py`, particiona por data e por Cliente, seleciona as features e
ajusta o pré-processador **apenas no treino**. É a única fonte de `X` e `y` do notebook: todo
candidato recebe exatamente a mesma matriz. As datas de corte vêm do registro de decisão do #127.

In [ ]:
SEMENTE = 42
CORTE_VALIDACAO = "2025-07-01"
CORTE_TESTE = "2026-01-01"

preparo = preparar_matriz(df, corte_validacao=CORTE_VALIDACAO, corte_teste=CORTE_TESTE)

X_treino, y_treino = preparo["matrizes"]["treino"], preparo["y"]["treino"]
X_validacao, y_validacao = preparo["matrizes"]["validacao"], preparo["y"]["validacao"]
grupos_treino = preparo["grupos"]["treino"]

for nome in ("treino", "validacao", "teste"):
    m = preparo["matrizes"][nome]
    print(f"{nome}: {m.shape[0]:,} linhas x {m.shape[1]} colunas | "
          f"prevalência de Detrator {preparo['y'][nome].mean():.4f}")

## 4. Registro dos candidatos

`modelos` recebe apenas candidatos já treinados com a matriz de treino. Enquanto estiver vazio,
as seções 6 e 7 executam sem comparar resultados.

In [ ]:
modelos = {}

## 5. Modelos candidatos

Cada candidato ganha uma subseção com descrição, justificativa, parâmetros, treinamento e
registro em `modelos`. Por ora, a seção 5.1 apenas importa e instancia Extra Trees; treinamento
e registro ficam para o card próprio.

### 5.1. Extra Trees

Extra Trees permite testar, na mesma matriz com variáveis numéricas e categóricas codificadas,
interações e relações não lineares que um modelo linear pode não capturar. A aleatorização dos
limiares de divisão oferece uma alternativa aos outros ensembles para a comparação na validação;
isso é uma hipótese de modelagem, não um resultado de desempenho.

Na instanciação atual, `random_state=SEMENTE` (42) fixa a aleatoriedade e `n_jobs=-1` usa os
núcleos disponíveis. Os demais hiperparâmetros seguem os padrões do scikit-learn: 100 árvores
(`n_estimators=100`), divisões por Gini (`criterion='gini'`), `max_features='sqrt'`,
`bootstrap=False` e `class_weight=None`. O efeito do desbalanceamento do alvo e a escolha de
`class_weight` serão examinados na validação, sem presumir ganho na classe minoritária.

In [ ]:
# Instanciação sem treino: o card de treinamento ajustará o modelo apenas em X_treino.
modelo_extra_trees = ExtraTreesClassifier(random_state=SEMENTE, n_jobs=-1)

## 6. Avaliação

Todos os candidatos são medidos na partição de **validação**, com as métricas do #105
(`metricas_no_teste` recebe a partição por parâmetro). O teste fica reservado ao modelo final.

In [ ]:
if modelos:
    resultados = metricas_no_teste(modelos, X_validacao, y_validacao)
    display(resultados.sort_values(resultados.columns[0], ascending=False))
else:
    resultados = pd.DataFrame()
    print("Nenhum candidato registrado em `modelos`.")

## 7. Visualização

In [ ]:
if resultados.empty:
    print("Sem resultados para visualizar.")
else:
    eixos = resultados.plot.bar(subplots=True, layout=(1, len(resultados.columns)),
                                figsize=(4 * len(resultados.columns), 4), legend=False, rot=0)
    plt.suptitle("Comparação dos candidatos na validação")
    plt.tight_layout()
    plt.show()

## 8. Conclusão

*Preencher após a definição e avaliação dos candidatos: qual modelo segue para o teste e por quê.*